# CareLoop — Part 4: structured follow-up review
Install `google-genai` and `pydantic` in Colab. The API key is entered at run time and is not stored in the notebook.

In [ ]:
import os
!git clone --depth 1 https://github.com/dhimanlalit420-cyber/careloop-risk-analysis.git
os.chdir('careloop-risk-analysis')
print('Repository data ready:', os.path.exists('appointments_clean.csv'), os.path.exists('data/followup_notes.csv'))


In [ ]:
!pip install -q google-genai pydantic
import json, time, pandas as pd
from google import genai
from google.genai import types
from pydantic import BaseModel, Field, ValidationError
API_KEY = input('Paste your Gemini API key: ')
client = genai.Client(api_key=API_KEY)
MODEL_NAME = 'gemini-3.8-flash'
SCHEMA = {'type':'object','properties':{'root_cause':{'type':'string'},'confidence':{'type':'number'},'summary':{'type':'string'},'recommended_action':{'type':'string'}},'required':['root_cause','confidence','summary','recommended_action']}
SYSTEM_TEXT = 'You extract the main reason a patient missed an outpatient appointment. Follow the requested JSON format.'
ALLOWED = ['forgot_or_no_reminder','distance_or_transport','cost_or_fee','illness_or_emergency','work_or_schedule','other']
CRAFT = '''Context: These are short call-centre notes after missed outpatient appointments in India.
Role: You are a careful note-classification assistant.
Action: Choose exactly one root cause from: forgot_or_no_reminder, distance_or_transport, cost_or_fee, illness_or_emergency, work_or_schedule, other. If evidence is ambiguous or does not fit, choose other rather than guess. Give confidence from 0 to 1, a summary of at most 25 words, and a practical recommended action.
Tone: Neutral, concise and nonjudgmental.
Format: Return only the requested JSON object with root_cause, confidence, summary, recommended_action.
Note: '''
def prompt_for(note_text): return CRAFT + str(note_text)
api_calls = 0
class InvalidJSONReply(Exception):
    def __init__(self, raw_response, message):
        self.raw_response = raw_response
        super().__init__(message)

def _request_text(note_text, prompt=None):
    global api_calls
    response = client.models.generate_content(model=MODEL_NAME,contents=prompt or prompt_for(note_text),config=types.GenerateContentConfig(system_instruction=SYSTEM_TEXT,temperature=0,max_output_tokens=1000,response_mime_type='application/json',response_schema=SCHEMA,thinking_config=types.ThinkingConfig(thinking_level='minimal')))
    api_calls += 1
    return response.text

def analyse_note(note_text, prompt=None):
    """Return the parsed four-field dictionary for one note."""
    data = json.loads(_request_text(note_text, prompt))
    return {field: data[field] for field in ['root_cause','confidence','summary','recommended_action']}

def analyse_note_with_raw(note_text, prompt=None):
    raw_response = _request_text(note_text, prompt)
    try:
        return json.loads(raw_response), raw_response
    except json.JSONDecodeError as exc:
        raise InvalidJSONReply(raw_response, str(exc)) from exc

first = analyse_note(pd.read_csv('data/followup_notes.csv').iloc[0].note_text)
print(first)


## 4.2 Validate and process first ten notes

In [ ]:
class FollowupInsight(BaseModel):
    root_cause: str = Field(min_length=3)
    confidence: float = Field(ge=0,le=1)
    summary: str = Field(min_length=5)
    recommended_action: str = Field(min_length=5)

def validate_insight(data):
    insight=FollowupInsight(**data)
    if insight.root_cause not in ALLOWED:
        raise ValueError(f'root_cause is outside allowed values: {insight.root_cause}')
    return insight
notes=pd.read_csv('data/followup_notes.csv').head(10)
results=[]
# api_calls continues from Task 4.1; requests are counted inside _request_text().
for _,row in notes.iterrows():
    text=row['note_text']
    base={'note_id':row.note_id,'appointment_id':row.appointment_id,'root_cause':'','confidence':'','summary':'','recommended_action':'','validation_status':'','raw_response':''}
    if pd.isna(text) or str(text).strip()=='':
        base.update(root_cause='other',confidence=0.0,summary='No note recorded',recommended_action='Re-call patient',validation_status='skipped_blank')
        results.append(base); continue
    try:
        data, raw_response=analyse_note_with_raw(text)
        base['raw_response']=raw_response
        for key in ['root_cause','confidence','summary','recommended_action']:
            if key in data: base[key]=data[key]
        validate_insight(data)
        base['validation_status']='valid'
    except Exception as exc:
        base['validation_status']='invalid'
        base['raw_response']=getattr(exc,'raw_response',base['raw_response'])
        print(row.note_id, type(exc).__name__, str(exc))
    results.append(base)
    time.sleep(13)
notes_analysed=pd.DataFrame(results,columns=['note_id','appointment_id','root_cause','confidence','summary','recommended_action','validation_status','raw_response'])
notes_analysed.to_csv('notes_analysed.csv',index=False)
print(notes_analysed[['note_id','root_cause','validation_status']])


## 4.3 Deliberate validation failures and prompt comparison

In [ ]:
bad_replies=[{'root_cause':'other','confidence':1.7,'summary':'Bad confidence example','recommended_action':'Call back'},
             {'root_cause':'other','confidence':0.5,'summary':'Missing action field'}]
for reply in bad_replies:
    try: FollowupInsight(**reply)
    except ValidationError as exc: print(exc)
# Compare a minimal prompt and the full CRAFT prompt. Both replies are checked against the allowed list.
by_note=notes.set_index('note_id')
comparison=[]
for note_id in ['N007','N009']:
    note_text=by_note.loc[note_id,'note_text']
    for version in ['V1','V2']:
        prompt=('Classify this note: ' + str(note_text)) if version=='V1' else prompt_for(note_text)
        data, raw_text=analyse_note_with_raw(note_text,prompt=prompt)
        cause=data.get('root_cause','')
        comparison.append({'note_id':note_id,'prompt':version,'root_cause':cause,'allowed':cause in ALLOWED,'raw_response':raw_text})
    time.sleep(13)
print('API calls across Tasks 4.1–4.3:',api_calls)


## 4.4 Join notes to cleaned bookings

In [ ]:
clean=pd.read_csv('appointments_clean.csv')
merged=notes_analysed.merge(clean[['appointment_id','reminder_sent','no_show']],on='appointment_id',how='left')
print('Merged rows:',len(merged))
print(pd.crosstab(merged.root_cause,merged.reminder_sent))
